# Carregamento de Dados de Inadimplência

Carrega o arquivo `desafio_inadimplencia_caixa.xlsx` do volume do Unity Catalog `inadimplencia_caixa.excel_dados.inadimplencia` em DataFrames Spark, com inspeção de abas, esquema e visualização dos dados.

In [0]:
%pip install openpyxl

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd

XLSX_PATH = "/Volumes/inadimplencia_caixa/excel_dados/inadimplencia/desafio_inadimplencia_caixa.xlsx"

xls = pd.ExcelFile(XLSX_PATH)
print("Abas encontradas:", xls.sheet_names)

for sheet in xls.sheet_names:
    pdf = pd.read_excel(XLSX_PATH, sheet_name=sheet)
    print(f"\n--- Aba '{sheet}': {pdf.shape[0]} linhas x {pdf.shape[1]} colunas ---")
    print("Colunas:", list(pdf.columns))

Abas encontradas: ['contratos', 'parcelas', 'indicadores_macro']

--- Aba 'contratos': 9999 linhas x 12 colunas ---
Colunas: ['id_contrato', 'dt_contratacao', 'vr_financiado', 'vr_entrada', 'tx_juros_anual', 'prazo_meses', 'faixa_renda', 'uf', 'tipo_imovel', 'programa_social', 'score_credito_contratacao', 'ltv']

--- Aba 'parcelas': 213701 linhas x 7 colunas ---
Colunas: ['id_contrato', 'dt_referencia', 'vr_parcela_devida', 'vr_pago', 'dias_atraso', 'saldo_devedor', 'fl_inadimplente']

--- Aba 'indicadores_macro': 36 linhas x 6 colunas ---
Colunas: ['dt_referencia', 'selic_12m', 'ipca_12m', 'tx_desemprego_12m', 'rendimento_medio', 'confianca_consumidor']


In [0]:
from pyspark.sql import functions as F

# Carrega cada aba do Excel em um DataFrame Spark
abas = {sheet: spark.createDataFrame(pd.read_excel(XLSX_PATH, sheet_name=sheet)) for sheet in xls.sheet_names}

df_contratos = abas["contratos"]
df_parcelas = abas["parcelas"]
df_indicadores_macro = abas["indicadores_macro"]

# Garante os tipos de data corretos
df_contratos = df_contratos.withColumn("dt_contratacao", F.to_date("dt_contratacao"))
df_parcelas = df_parcelas.withColumn("dt_referencia", F.to_date("dt_referencia"))
df_indicadores_macro = df_indicadores_macro.withColumn("dt_referencia", F.to_date("dt_referencia"))

for nome, df in [("df_contratos", df_contratos), ("df_parcelas", df_parcelas), ("df_indicadores_macro", df_indicadores_macro)]:
    print(f"{nome}: {df.count()} linhas")
    df.printSchema()
    print()

df_contratos: 9999 linhas
root
 |-- id_contrato: long (nullable = true)
 |-- dt_contratacao: date (nullable = true)
 |-- vr_financiado: double (nullable = true)
 |-- vr_entrada: double (nullable = true)
 |-- tx_juros_anual: double (nullable = true)
 |-- prazo_meses: long (nullable = true)
 |-- faixa_renda: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- tipo_imovel: string (nullable = true)
 |-- programa_social: string (nullable = true)
 |-- score_credito_contratacao: long (nullable = true)
 |-- ltv: double (nullable = true)


df_parcelas: 213701 linhas
root
 |-- id_contrato: long (nullable = true)
 |-- dt_referencia: date (nullable = true)
 |-- vr_parcela_devida: double (nullable = true)
 |-- vr_pago: double (nullable = true)
 |-- dias_atraso: long (nullable = true)
 |-- saldo_devedor: double (nullable = true)
 |-- fl_inadimplente: long (nullable = true)


df_indicadores_macro: 36 linhas
root
 |-- dt_referencia: date (nullable = true)
 |-- selic_12m: double (nullable =

In [0]:
display(df_contratos.limit(5))
display(df_parcelas.limit(5))
display(df_indicadores_macro.limit(5))

id_contrato,dt_contratacao,vr_financiado,vr_entrada,tx_juros_anual,prazo_meses,faixa_renda,uf,tipo_imovel,programa_social,score_credito_contratacao,ltv
1,2022-01-01,318125.78,132286.64,9.47,240,Acima de 10 SM,PA,Usado,FGTS,801,70.63
2,2022-01-01,331881.9,124846.24,7.98,360,Acima de 10 SM,PR,Usado,FGTS,814,72.67
3,2022-01-01,134318.01,28375.09,5.79,360,3 a 6 SM,MS,Usado,MCMV,624,82.56
4,2022-01-01,388430.1,85697.76,9.59,180,6 a 10 SM,AM,Usado,Livre,756,81.93
5,2022-01-01,164952.84,28489.66,8.74,180,Até 3 SM,RJ,Novo,FGTS,508,85.27


id_contrato,dt_referencia,vr_parcela_devida,vr_pago,dias_atraso,saldo_devedor,fl_inadimplente
1,2023-03-01,3102.73,14.29,94,321372.86,1
1,2024-03-01,3102.73,213.86,140,319499.88,1
1,2022-02-01,3102.73,3102.73,0,324197.57,0
1,2022-03-01,3102.73,3102.73,0,323758.66,0
1,2022-04-01,3102.73,3102.73,0,323316.15,0


dt_referencia,selic_12m,ipca_12m,tx_desemprego_12m,rendimento_medio,confianca_consumidor
2022-01-01,9.27,10.56,11.49,2785.15,82.39
2022-02-01,10.74,9.74,11.71,2791.08,85.8
2022-03-01,11.78,9.76,10.86,2766.77,84.81
2022-04-01,12.83,10.04,11.05,2817.18,85.73
2022-05-01,13.24,10.04,10.61,2831.49,88.41
